In [1]:
from numcosmo_py.catalog import MockGenerator
from numcosmo_py import Nc, Ncm
from numcosmo_py.catalog import sky_match
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.table import unique, Table
from  run_confusion_matrix import calculate_split_metrics, calculate_catalog_metrics
from matplotlib import pyplot as plt
Ncm.cfg_init()
%load_ext autoreload
%autoreload 2

In [2]:
Omega_b = 0.0486
Omega_c = 0.2614
Omega_k = 0.0
H0 = 67.7

cosmo = Nc.HICosmoDEXcdm.new()
cosmo.omega_x2omega_k()
cosmo["Omegab"] = Omega_b
cosmo["Omegac"] = Omega_c
cosmo["Omegak"] = Omega_k
cosmo["H0"] = H0
cosmo["w"] = -1.0

prim = Nc.HIPrimPowerLaw.new()
prim.param_set_by_name("ln10e10ASA", 3.0)
prim.props.n_SA =  0.963 

reion = Nc.HIReionCamb.new()

cosmo.add_submodel(prim)
cosmo.add_submodel(reion)

tf = Nc.TransferFuncEH()

psml = Nc.PowspecMLTransfer.new(tf)
psml.require_kmin(1.0e-6)
psml.require_kmax(1.0e3)


psf = Ncm.PowspecFilter.new(psml, Ncm.PowspecFilterType.TOPHAT)
psf.set_best_lnr0()
psf.prepare(cosmo)

old_amplitude = np.exp(prim.props.ln10e10ASA)
ln10e10ASA = np.log((0.8/ cosmo.sigma8(psf)) ** 2 * old_amplitude)
prim.param_set_desc("ln10e10ASA", {"lower-bound": 2.0,"upper-bound": 5.0,"scale": 1.0,"abstol": 1.0e-50,"fit": True,"value": float(ln10e10ASA)})

dist = Nc.Distance.new(2.0)
dist.prepare(cosmo)

mulf = Nc.MultiplicityFuncDespali.new()
mulf.set_mdef(Nc.MultiplicityFuncMassDef.VIRIAL)
hmf = Nc.HaloMassFunction.new(dist, psf, mulf)

cluster_m = Nc.ClusterMassAscaso(lnRichness_min = np.log(1) ,lnRichness_max = np.log(1000), M0=1e13)
cluster_m.param_set_by_name("cut", np.log(1))
cluster_m.param_set_desc("mup0",{"fit": True,"value": 3.022142935})
cluster_m.param_set_desc("mup1",{"fit": True,"value": 1.25})
cluster_m.param_set_desc("mup2",{"fit": True,"value": 0.0})
cluster_m.param_set_desc("sigmap0",{"fit": True,"value": 0.5})
cluster_m.param_set_desc("sigmap1",{"fit": True,"value": 0.0})
cluster_m.param_set_desc("sigmap2",{"fit": True,"value": 0.0})
cluster_m = Nc.ClusterMassLnnormal(lnMobs_min=np.log(1e12), lnMobs_max=np.log(1e16),bias=0,sigma=0.1)

In [3]:
class Completeness_model:
    
    """Class to generate completeness model for halo/cluster mocks."""
    def __init__(self, a):
        self.a = a

    
    def complete(self, mass, z):
        return np.ones_like(mass)

    def incomplete(self, mass, z):
        return  np.full_like(mass, self.a, dtype=float)
    
comp= Completeness_model(0.7)
mock_test = MockGenerator(cosmo=cosmo, hmf=hmf, cluster_m=cluster_m,halo_mass_interval=[10**(12.0),1e16])
print(mock_test.sky_area())
halos_mock = mock_test.generate_halos_from_hmf(comp.incomplete)
print(len(halos_mock['is_detected']))
print(len(halos_mock[halos_mock['is_detected'] == 1]['z']))
print(len(halos_mock[halos_mock['is_detected'] == 0]['z']))
print(len(halos_mock[halos_mock['is_detected'] == 1]['z'])/len(halos_mock['is_detected']))

401.93756131445895
503870
353320
150550
0.7012126143648163


In [ ]:
mock_test.cluster_mass_interval = [10**(12.0),1e15]
halos_mock["Mass_obs"] = halos_mock["Mass_obs"]
class Purity_model:
    
    """Class to generate completeness model for halo/cluster mocks."""
    def __init__(self, b):
        self.b = b

    
    def pure(self, mass, z):
        return np.ones_like(mass)
        
    def impure(self, mass, z):
        return  np.full_like(mass, self.b, dtype=float)

def scaling_relation(lnMobs, z):
    return lnMobs

pur= Purity_model(0.8)
clusters_mock = mock_test.generate_clusters_from_halos(halos_mock,2.0,pur.impure,scaling_relation)
#clusters_mock = mock_test.generate_clusters_from_halos(halos_mock,2.0,None,scaling_relation)
print(len(clusters_mock['parent_id']))
print(len(clusters_mock[clusters_mock['parent_id'] != 0]['z']))
print(len(clusters_mock[clusters_mock['parent_id'] == 0]['z']))
print(len(clusters_mock[clusters_mock['parent_id'] != 0]['z'])/len(clusters_mock['parent_id'] ))

# Confusion Matrix DistanceMethod

In [5]:
halo_coordinates = {"RA":"RA" , "DEC":"DEC" , "z":"z"}
halo_properties  = {"Mass":"halo_mass","halo_id":"halo_id", "R200c":"R200", 'is_detected': 'is_detected'}
detections_coordinates =  {"RA":"RA" , "DEC":"DEC" , "z":"z"}
detections_properties  = {"Mass":"cluster_mass","cluster_id":"cluster_id", "R200c":"R200", "parent_id": "parent_id"}

results_data = []


distance_methods = [sky_match.DistanceMethod.QUERY_RADIUS, sky_match.DistanceMethod.MATCH_RADIUS, 
                    sky_match.DistanceMethod.MIN_RADIUS,sky_match.DistanceMethod.MAX_RADIUS, sky_match.DistanceMethod.ANGULAR_SEPARATION]

#distance_methods = [sky_match.DistanceMethod.QUERY_RADIUS, sky_match.DistanceMethod.MATCH_RADIUS]
def get_ratios(metrics_result):
    """Safely extracts TP/FP/TN/FN even if the metrics function returns an error string."""
    if isinstance(metrics_result, dict) and 'ratios' in metrics_result:
        return metrics_result['ratios']
    # Returns NaN if metrics fail or return a string, ensuring the code doesn't crash
    return {'TP': np.nan, 'FP': np.nan, 'TN': np.nan, 'FN': np.nan}

# --- Initialization ---
dfs_by_method = {}
halos_mocks_mass = np.log(np.array([1e12, 10**(12.5), 10**(12.7), 10**(12.9), 10**(13.0), 10**(13.3), 10**(13.6), 10**(14.0)]))
#halos_mocks_mass = np.log(np.array([10**(13.3), 10**(13.6), 10**(14.0)]))

# --- Main Loops ---
for d_method in distance_methods:
    # Identify the method name for the dictionary key
    method_name = d_method.name if hasattr(d_method, 'name') else str(d_method)
    current_method_rows = []
    
    print(f"Starting analysis for method: {method_name}")

    for mass in halos_mocks_mass:
        # 1. Data Prep
        halos_mock_cutted = halos_mock[halos_mock['Mass'] >= mass]
        current_halo_size = len(halos_mock_cutted['Mass'])
        
        # 2. SkyMatch Initialization
        halos = sky_match.SkyMatch(
                    query_data=halos_mock_cutted, 
                    query_coordinates=halo_coordinates,
                    match_data=clusters_mock,
                    match_coordinates=detections_coordinates
                )
        detections = halos.invert_query_match()
        
        # 3. Matching Phase
        halos_matched = halos.match_2d(cosmo, 50, distance_method=d_method)
        detections_matched = detections.match_2d(cosmo, 50, distance_method=d_method)
        
        # 4. Filtering & Best Match Selection
        mask_halos = (halos_matched.filter_mask_by_redshift_proximity(sigma_z=0.01) & halos_matched.filter_mask_by_distance(2))
        unique_halos = halos_matched.select_best(mask=mask_halos, selection_criteria=sky_match.SelectionCriteria.DISTANCES, more_massive_column='Mass')
        
        mask_detections = (detections_matched.filter_mask_by_redshift_proximity(sigma_z=0.01) & detections_matched.filter_mask_by_distance(2))
        unique_detections = detections_matched.select_best(mask=mask_detections, selection_criteria=sky_match.SelectionCriteria.DISTANCES, more_massive_column='Mass')
        
        # 5. Table Generation
        multiple_halos_table = halos_matched.to_table_complete(mask=mask_halos, query_properties=halo_properties, match_properties=detections_properties)
        multiple_detections_table = detections_matched.to_table_complete(mask=mask_detections, query_properties=detections_properties, match_properties=halo_properties)
        
        unique_halos_table = halos_matched.to_table_best(best=unique_halos, query_properties=halo_properties, match_properties=detections_properties)
        unique_detections_table = detections_matched.to_table_best(best=unique_detections, query_properties=detections_properties, match_properties=halo_properties)
        
        # 6. Optimized Cross-Match phase (Vectorized)
        cross_indices = unique_halos.get_cross_match_indices(unique_detections)
        cross = unique_halos_table[np.isin(unique_halos_table['Index'], cross_indices)]
        
        # 7. Metrics Calculation (with get_ratios safety wrapper)
        m_halo_r = get_ratios(calculate_split_metrics(multiple_halos_table, 'halo'))
        m_clus_r = get_ratios(calculate_split_metrics(multiple_detections_table, 'cluster'))
        
        u_halo_r = get_ratios(calculate_catalog_metrics(unique_halos_table, halos_mock, 'halo'))
        u_clus_r = get_ratios(calculate_catalog_metrics(unique_detections_table, clusters_mock, 'cluster'))
        
        c_halo_r = get_ratios(calculate_catalog_metrics(cross, halos_mock, 'halo'))
        c_clus_r = get_ratios(calculate_catalog_metrics(cross, clusters_mock, 'cluster'))

        # 8. Row Construction
        row = {
            'mass_limit': np.exp(mass),
            'halo_size': current_halo_size,
            # Multiples
            'm_halo_TP': m_halo_r['TP'], 'm_halo_FP': m_halo_r['FP'], 'm_halo_TN': m_halo_r['TN'], 'm_halo_FN': m_halo_r['FN'],
            'm_clus_TP': m_clus_r['TP'], 'm_clus_FP': m_clus_r['FP'], 'm_clus_TN': m_clus_r['TN'], 'm_clus_FN': m_clus_r['FN'],
            # Uniques
            'u_halo_TP': u_halo_r['TP'], 'u_halo_FP': u_halo_r['FP'], 'u_halo_TN': u_halo_r['TN'], 'u_halo_FN': u_halo_r['FN'],
            'u_clus_TP': u_clus_r['TP'], 'u_clus_FP': u_clus_r['FP'], 'u_clus_TN': u_clus_r['TN'], 'u_clus_FN': u_clus_r['FN'],
            # Cross
            'c_halo_TP': c_halo_r['TP'], 'c_halo_FP': c_halo_r['FP'], 'c_halo_TN': c_halo_r['TN'], 'c_halo_FN': c_halo_r['FN'],
            'c_clus_TP': c_clus_r['TP'], 'c_clus_FP': c_clus_r['FP'], 'c_clus_TN': c_clus_r['TN'], 'c_clus_FN': c_clus_r['FN']
        }
        
        current_method_rows.append(row)
        print(f"  > Finished M = {np.exp(mass):.2e}")

    # Save the current method's list of rows as a DataFrame
    dfs_by_method[method_name] = pd.DataFrame(current_method_rows)

# --- Final Output ---
print("\nAll methods complete. Access them via dfs_by_method['MethodName']")

hdul = fits.HDUList([fits.PrimaryHDU()]) # Empty primary HDU

for method_name, df in dfs_by_method.items():
    hdu = fits.BinTableHDU(Table.from_pandas(df))
    hdu.name = method_name
    hdul.append(hdu)

hdul.writeto('distance_methods_multihdu.fits', overwrite=True)

Starting analysis for method: QUERY_RADIUS
  > Finished M = 1.00e+12
  > Finished M = 3.16e+12
  > Finished M = 5.01e+12
  > Finished M = 7.94e+12
  > Finished M = 1.00e+13
  > Finished M = 2.00e+13
  > Finished M = 3.98e+13
  > Finished M = 1.00e+14
Starting analysis for method: MATCH_RADIUS
  > Finished M = 1.00e+12
  > Finished M = 3.16e+12
  > Finished M = 5.01e+12
  > Finished M = 7.94e+12
  > Finished M = 1.00e+13
  > Finished M = 2.00e+13
  > Finished M = 3.98e+13
  > Finished M = 1.00e+14
Starting analysis for method: MIN_RADIUS
  > Finished M = 1.00e+12
  > Finished M = 3.16e+12
  > Finished M = 5.01e+12
  > Finished M = 7.94e+12
  > Finished M = 1.00e+13
  > Finished M = 2.00e+13
  > Finished M = 3.98e+13
  > Finished M = 1.00e+14
Starting analysis for method: MAX_RADIUS
  > Finished M = 1.00e+12
  > Finished M = 3.16e+12
  > Finished M = 5.01e+12
  > Finished M = 7.94e+12
  > Finished M = 1.00e+13
  > Finished M = 2.00e+13
  > Finished M = 3.98e+13
  > Finished M = 1.00e+14


# Confusion Matrix SelectionCriteria

In [9]:
def get_ratios(metrics_result):
    """Safely extracts TP/FP/TN/FN even if the metrics function returns an error string."""
    if isinstance(metrics_result, dict) and 'ratios' in metrics_result:
        return metrics_result['ratios']
    return {'TP': np.nan, 'FP': np.nan, 'TN': np.nan, 'FN': np.nan}

# --- Configuration ---
selection_criterium = [
    sky_match.SelectionCriteria.DISTANCES, 
    sky_match.SelectionCriteria.REDSHIFT_PROXIMITY, 
    sky_match.SelectionCriteria.MORE_MASSIVE
]

dfs_by_criteria = {}
halos_mocks_mass = np.log(np.array([1e12, 10**(12.5), 10**(12.7), 10**(12.9), 10**(13.0), 10**(13.3), 10**(13.6), 10**(14.0)]))

# --- Main Loops ---
for criteria in selection_criterium:
    # Identify the criteria name for the dictionary key and FITS HDU
    criteria_name = criteria.name if hasattr(criteria, 'name') else str(criteria)
    current_criteria_rows = []
    
    print(f"Starting analysis for selection criteria: {criteria_name}")

    for mass in halos_mocks_mass:
        # 1. Data Prep
        halos_mock_cutted = halos_mock[halos_mock['Mass'] >= mass]
        current_halo_size = len(halos_mock_cutted['Mass'])
        
        # 2. SkyMatch Initialization
        halos = sky_match.SkyMatch(
                    query_data=halos_mock_cutted, 
                    query_coordinates=halo_coordinates,
                    match_data=clusters_mock,
                    match_coordinates=detections_coordinates
                )
        detections = halos.invert_query_match()
        
        # 3. Matching Phase
        halos_matched = halos.match_2d(cosmo, 50, distance_method=sky_match.DistanceMethod.QUERY_RADIUS)
        detections_matched = detections.match_2d(cosmo, 50, distance_method=sky_match.DistanceMethod.MATCH_RADIUS)
        
        # 4. Filtering & Best Match Selection (Applying the current loop's criteria)
        mask_halos = (halos_matched.filter_mask_by_redshift_proximity(sigma_z=0.01) & halos_matched.filter_mask_by_distance(2))
        unique_halos = halos_matched.select_best(mask=mask_halos, selection_criteria=criteria, more_massive_column='Mass')
        
        mask_detections = (detections_matched.filter_mask_by_redshift_proximity(sigma_z=0.01) & detections_matched.filter_mask_by_distance(2))
        unique_detections = detections_matched.select_best(mask=mask_detections, selection_criteria=criteria, more_massive_column='Mass')
        
        # 5. Table Generation
        multiple_halos_table = halos_matched.to_table_complete(mask=mask_halos, query_properties=halo_properties, match_properties=detections_properties)
        multiple_detections_table = detections_matched.to_table_complete(mask=mask_detections, query_properties=detections_properties, match_properties=halo_properties)
        
        unique_halos_table = halos_matched.to_table_best(best=unique_halos, query_properties=halo_properties, match_properties=detections_properties)
        unique_detections_table = detections_matched.to_table_best(best=unique_detections, query_properties=detections_properties, match_properties=halo_properties)
        
        # 6. Optimized Cross-Match phase
        cross_indices = unique_halos.get_cross_match_indices(unique_detections)
        cross = unique_halos_table[np.isin(unique_halos_table['Index'], cross_indices)]
        
        # 7. Metrics Calculation
        m_halo_r = get_ratios(calculate_split_metrics(multiple_halos_table, 'halo'))
        m_clus_r = get_ratios(calculate_split_metrics(multiple_detections_table, 'cluster'))
        u_halo_r = get_ratios(calculate_catalog_metrics(unique_halos_table, halos_mock, 'halo'))
        u_clus_r = get_ratios(calculate_catalog_metrics(unique_detections_table, clusters_mock, 'cluster'))
        c_halo_r = get_ratios(calculate_catalog_metrics(cross, halos_mock, 'halo'))
        c_clus_r = get_ratios(calculate_catalog_metrics(cross, clusters_mock, 'cluster'))

        # 8. Row Construction
        row = {
            'mass_limit': np.exp(mass),
            'halo_size': current_halo_size,
            'm_halo_TP': m_halo_r['TP'], 'm_halo_FP': m_halo_r['FP'], 'm_halo_TN': m_halo_r['TN'], 'm_halo_FN': m_halo_r['FN'],
            'm_clus_TP': m_clus_r['TP'], 'm_clus_FP': m_clus_r['FP'], 'm_clus_TN': m_clus_r['TN'], 'm_clus_FN': m_clus_r['FN'],
            'u_halo_TP': u_halo_r['TP'], 'u_halo_FP': u_halo_r['FP'], 'u_halo_TN': u_halo_r['TN'], 'u_halo_FN': u_halo_r['FN'],
            'u_clus_TP': u_clus_r['TP'], 'u_clus_FP': u_clus_r['FP'], 'u_clus_TN': u_clus_r['TN'], 'u_clus_FN': u_clus_r['FN'],
            'c_halo_TP': c_halo_r['TP'], 'c_halo_FP': c_halo_r['FP'], 'c_halo_TN': c_halo_r['TN'], 'c_halo_FN': c_halo_r['FN'],
            'c_clus_TP': c_clus_r['TP'], 'c_clus_FP': c_clus_r['FP'], 'c_clus_TN': c_clus_r['TN'], 'c_clus_FN': c_clus_r['FN']
        }
        
        current_criteria_rows.append(row)
        print(f"  > Finished M = {np.exp(mass):.2e}")

    # Store in dictionary
    dfs_by_criteria[criteria_name] = pd.DataFrame(current_criteria_rows)

# --- Save to Multi-HDU FITS ---
hdul = fits.HDUList([fits.PrimaryHDU()]) # Empty primary

for name, df in dfs_by_criteria.items():
    hdu = fits.BinTableHDU(Table.from_pandas(df))
    hdu.name = name
    hdul.append(hdu)

hdul.writeto('selection_criteria_results.fits', overwrite=True)
print("\nAll criteria complete. Results saved to 'selection_criteria_results.fits'")

Starting analysis for selection criteria: DISTANCES
  > Finished M = 1.00e+12
  > Finished M = 3.16e+12
  > Finished M = 5.01e+12
  > Finished M = 7.94e+12
  > Finished M = 1.00e+13
  > Finished M = 2.00e+13
  > Finished M = 3.98e+13
  > Finished M = 1.00e+14
Starting analysis for selection criteria: REDSHIFT_PROXIMITY
  > Finished M = 1.00e+12
  > Finished M = 3.16e+12
  > Finished M = 5.01e+12
  > Finished M = 7.94e+12
  > Finished M = 1.00e+13
  > Finished M = 2.00e+13
  > Finished M = 3.98e+13
  > Finished M = 1.00e+14
Starting analysis for selection criteria: MORE_MASSIVE
  > Finished M = 1.00e+12
  > Finished M = 3.16e+12
  > Finished M = 5.01e+12
  > Finished M = 7.94e+12
  > Finished M = 1.00e+13
  > Finished M = 2.00e+13
  > Finished M = 3.98e+13
  > Finished M = 1.00e+14

All criteria complete. Results saved to 'selection_criteria_results.fits'


# Confusion Matrix knn

In [ ]:
def get_ratios(metrics_result):
    """Safely extracts TP/FP/TN/FN even if the metrics function returns an error string."""
    if isinstance(metrics_result, dict) and 'ratios' in metrics_result:
        return metrics_result['ratios']
    return {'TP': np.nan, 'FP': np.nan, 'TN': np.nan, 'FN': np.nan}

knns = [10, 20, 50, 70, 100]
dfs_by_knn = {}
halos_mocks_mass = np.log(np.array([1e12, 10**(12.5), 10**(12.7), 10**(12.9), 10**(13.0), 10**(13.3), 10**(13.6), 10**(14.0)]))

# --- Main Loops ---
for k in knns:
    # Identify the knn name for the dictionary key and FITS HDU
    knn_label = f"KNN_{k}"
    current_knn_rows = []
    
    print(f"Starting analysis for KNN value: {k}")

    for mass in halos_mocks_mass:
        # 1. Data Prep
        halos_mock_cutted = halos_mock[halos_mock['Mass'] >= mass]
        current_halo_size = len(halos_mock_cutted['Mass'])
        
        # 2. SkyMatch Initialization
        halos = sky_match.SkyMatch(
                    query_data=halos_mock_cutted, 
                    query_coordinates=halo_coordinates,
                    match_data=clusters_mock,
                    match_coordinates=detections_coordinates
                )
        detections = halos.invert_query_match()
        
        # 3. Matching Phase (Passing the loop variable 'k' to knn parameter)
        halos_matched = halos.match_2d(cosmo, k, distance_method=sky_match.DistanceMethod.QUERY_RADIUS)
        detections_matched = detections.match_2d(cosmo, k, distance_method=sky_match.DistanceMethod.MATCH_RADIUS)
        
        # 4. Filtering & Best Match Selection
        mask_halos = (halos_matched.filter_mask_by_redshift_proximity(sigma_z=0.01) & halos_matched.filter_mask_by_distance(2))
        unique_halos = halos_matched.select_best(mask=mask_halos, selection_criteria=sky_match.SelectionCriteria.DISTANCES, more_massive_column='Mass')
        
        mask_detections = (detections_matched.filter_mask_by_redshift_proximity(sigma_z=0.01) & detections_matched.filter_mask_by_distance(2))
        unique_detections = detections_matched.select_best(mask=mask_detections, selection_criteria=sky_match.SelectionCriteria.DISTANCES, more_massive_column='Mass')
        
        # 5. Table Generation
        multiple_halos_table = halos_matched.to_table_complete(mask=mask_halos, query_properties=halo_properties, match_properties=detections_properties)
        multiple_detections_table = detections_matched.to_table_complete(mask=mask_detections, query_properties=detections_properties, match_properties=halo_properties)
        
        unique_halos_table = halos_matched.to_table_best(best=unique_halos, query_properties=halo_properties, match_properties=detections_properties)
        unique_detections_table = detections_matched.to_table_best(best=unique_detections, query_properties=detections_properties, match_properties=halo_properties)
        
        # 6. Optimized Cross-Match phase
        cross_indices = unique_halos.get_cross_match_indices(unique_detections)
        cross = unique_halos_table[np.isin(unique_halos_table['Index'], cross_indices)]
        
        # 7. Metrics Calculation
        m_halo_r = get_ratios(calculate_split_metrics(multiple_halos_table, 'halo'))
        m_clus_r = get_ratios(calculate_split_metrics(multiple_detections_table, 'cluster'))
        u_halo_r = get_ratios(calculate_catalog_metrics(unique_halos_table, halos_mock, 'halo'))
        u_clus_r = get_ratios(calculate_catalog_metrics(unique_detections_table, clusters_mock, 'cluster'))
        c_halo_r = get_ratios(calculate_catalog_metrics(cross, halos_mock, 'halo'))
        c_clus_r = get_ratios(calculate_catalog_metrics(cross, clusters_mock, 'cluster'))

        # 8. Row Construction
        row = {
            'mass_limit': np.exp(mass),
            'halo_size': current_halo_size,
            'knn_value': k,
            'm_halo_TP': m_halo_r['TP'], 'm_halo_FP': m_halo_r['FP'], 'm_halo_TN': m_halo_r['TN'], 'm_halo_FN': m_halo_r['FN'],
            'm_clus_TP': m_clus_r['TP'], 'm_clus_FP': m_clus_r['FP'], 'm_clus_TN': m_clus_r['TN'], 'm_clus_FN': m_clus_r['FN'],
            'u_halo_TP': u_halo_r['TP'], 'u_halo_FP': u_halo_r['FP'], 'u_halo_TN': u_halo_r['TN'], 'u_halo_FN': u_halo_r['FN'],
            'u_clus_TP': u_clus_r['TP'], 'u_clus_FP': u_clus_r['FP'], 'u_clus_TN': u_clus_r['TN'], 'u_clus_FN': u_clus_r['FN'],
            'c_halo_TP': c_halo_r['TP'], 'c_halo_FP': c_halo_r['FP'], 'c_halo_TN': c_halo_r['TN'], 'c_halo_FN': c_halo_r['FN'],
            'c_clus_TP': c_clus_r['TP'], 'c_clus_FP': c_clus_r['FP'], 'c_clus_TN': c_clus_r['TN'], 'c_clus_FN': c_clus_r['FN']
        }
        
        current_knn_rows.append(row)
        print(f"  > Finished M = {np.exp(mass):.2e}")

    # Store in dictionary
    dfs_by_knn[knn_label] = pd.DataFrame(current_knn_rows)

# --- Save to Multi-HDU FITS ---
hdul_knn = fits.HDUList([fits.PrimaryHDU()]) 

for label, df in dfs_by_knn.items():
    hdu = fits.BinTableHDU(Table.from_pandas(df))
    hdu.name = label
    hdul_knn.append(hdu)

hdul_knn.writeto('knn_results.fits', overwrite=True)
print("\nAll KNN values complete. Results saved to 'knn_results.fits'")

Starting analysis for KNN value: 10


# Confusion Matrix distance_mask

In [ ]:
def get_ratios(metrics_result):
    """Safely extracts TP/FP/TN/FN even if the metrics function returns an error string."""
    if isinstance(metrics_result, dict) and 'ratios' in metrics_result:
        return metrics_result['ratios']
    return {'TP': np.nan, 'FP': np.nan, 'TN': np.nan, 'FN': np.nan}
# --- Configuration ---
distances_masked = [1, 2, 5, 10, 20]
dfs_by_distance = {}
halos_mocks_mass = np.log(np.array([1e12, 10**(12.5), 10**(12.7), 10**(12.9), 10**(13.0), 10**(13.3), 10**(13.6), 10**(14.0)]))

# --- Main Loops ---
for d_mask in distances_masked:
    # Identify the distance name for the dictionary key and FITS HDU
    # Using 'D' prefix because HDU names cannot start with numbers easily in some viewers
    dist_label = f"DIST_{d_mask}MPC"
    current_dist_rows = []
    
    print(f"Starting analysis for Masking Distance: {d_mask} Mpc")

    for mass in halos_mocks_mass:
        # 1. Data Prep
        halos_mock_cutted = halos_mock[halos_mock['Mass'] >= mass]
        current_halo_size = len(halos_mock_cutted['Mass'])
        
        # 2. SkyMatch Initialization
        halos = sky_match.SkyMatch(
                    query_data=halos_mock_cutted, 
                    query_coordinates=halo_coordinates,
                    match_data=clusters_mock,
                    match_coordinates=detections_coordinates
                )
        detections = halos.invert_query_match()
        
        # 3. Matching Phase (Fixed at 50 neighbors or radius)
        halos_matched = halos.match_2d(cosmo, 50, distance_method=sky_match.DistanceMethod.QUERY_RADIUS)
        detections_matched = detections.match_2d(cosmo, 50, distance_method=sky_match.DistanceMethod.MATCH_RADIUS)
        
        # 4. Filtering & Best Match Selection (NOW USING d_mask variable)
        mask_halos = (halos_matched.filter_mask_by_redshift_proximity(sigma_z=0.01) & 
                      halos_matched.filter_mask_by_distance(d_mask))
        
        unique_halos = halos_matched.select_best(mask=mask_halos, 
                                               selection_criteria=sky_match.SelectionCriteria.DISTANCES, 
                                               more_massive_column='Mass')
        
        mask_detections = (detections_matched.filter_mask_by_redshift_proximity(sigma_z=0.01) & 
                           detections_matched.filter_mask_by_distance(d_mask))
        
        unique_detections = detections_matched.select_best(mask=mask_detections, 
                                                     selection_criteria=sky_match.SelectionCriteria.DISTANCES, 
                                                     more_massive_column='Mass')
        
        # 5. Table Generation
        multiple_halos_table = halos_matched.to_table_complete(mask=mask_halos, query_properties=halo_properties, match_properties=detections_properties)
        multiple_detections_table = detections_matched.to_table_complete(mask=mask_detections, query_properties=detections_properties, match_properties=halo_properties)
        
        unique_halos_table = halos_matched.to_table_best(best=unique_halos, query_properties=halo_properties, match_properties=detections_properties)
        unique_detections_table = detections_matched.to_table_best(best=unique_detections, query_properties=detections_properties, match_properties=halo_properties)
        
        # 6. Optimized Cross-Match phase
        cross_indices = unique_halos.get_cross_match_indices(unique_detections)
        cross = unique_halos_table[np.isin(unique_halos_table['Index'], cross_indices)]
        
        # 7. Metrics Calculation
        m_halo_r = get_ratios(calculate_split_metrics(multiple_halos_table, 'halo'))
        m_clus_r = get_ratios(calculate_split_metrics(multiple_detections_table, 'cluster'))
        u_halo_r = get_ratios(calculate_catalog_metrics(unique_halos_table, halos_mock, 'halo'))
        u_clus_r = get_ratios(calculate_catalog_metrics(unique_detections_table, clusters_mock, 'cluster'))
        c_halo_r = get_ratios(calculate_catalog_metrics(cross, halos_mock, 'halo'))
        c_clus_r = get_ratios(calculate_catalog_metrics(cross, clusters_mock, 'cluster'))

        # 8. Row Construction
        row = {
            'mass_limit': np.exp(mass),
            'halo_size': current_halo_size,
            'mask_dist': d_mask,
            'm_halo_TP': m_halo_r['TP'], 'm_halo_FP': m_halo_r['FP'], 'm_halo_TN': m_halo_r['TN'], 'm_halo_FN': m_halo_r['FN'],
            'm_clus_TP': m_clus_r['TP'], 'm_clus_FP': m_clus_r['FP'], 'm_clus_TN': m_clus_r['TN'], 'm_clus_FN': m_clus_r['FN'],
            'u_halo_TP': u_halo_r['TP'], 'u_halo_FP': u_halo_r['FP'], 'u_halo_TN': u_halo_r['TN'], 'u_halo_FN': u_halo_r['FN'],
            'u_clus_TP': u_clus_r['TP'], 'u_clus_FP': u_clus_r['FP'], 'u_clus_TN': u_clus_r['TN'], 'u_clus_FN': u_clus_r['FN'],
            'c_halo_TP': c_halo_r['TP'], 'c_halo_FP': c_halo_r['FP'], 'c_halo_TN': c_halo_r['TN'], 'c_halo_FN': c_halo_r['FN'],
            'c_clus_TP': c_clus_r['TP'], 'c_clus_FP': c_clus_r['FP'], 'c_clus_TN': c_clus_r['TN'], 'c_clus_FN': c_clus_r['FN']
        }
        
        current_dist_rows.append(row)
        print(f"  > Finished M = {np.exp(mass):.2e}")

    # Store in dictionary
    dfs_by_distance[dist_label] = pd.DataFrame(current_dist_rows)

# --- Save to Multi-HDU FITS ---
hdul_dist = fits.HDUList([fits.PrimaryHDU()]) 

for label, df in dfs_by_distance.items():
    hdu = fits.BinTableHDU(Table.from_pandas(df))
    hdu.name = label
    hdul_dist.append(hdu)

hdul_dist.writeto('masking_distance_results.fits', overwrite=True)
print("\nAll distances complete. Results saved to 'masking_distance_results.fits'")

# Confusion Matrix sigma_z mask

In [ ]:
def get_ratios(metrics_result):
    """Safely extracts TP/FP/TN/FN even if the metrics function returns an error string."""
    if isinstance(metrics_result, dict) and 'ratios' in metrics_result:
        return metrics_result['ratios']
    return {'TP': np.nan, 'FP': np.nan, 'TN': np.nan, 'FN': np.nan}

# --- Configuration ---
sigma_zs = [0.01, 0.05, 0.07, 0.1]
dfs_by_sigma = {}
halos_mocks_mass = np.log(np.array([1e12, 10**(12.5), 10**(12.7), 10**(12.9), 10**(13.0), 10**(13.3), 10**(13.6), 10**(14.0)]))

# --- Main Loops ---
for s_z in sigma_zs:
    # Identify the sigma_z name for the dictionary key and FITS HDU
    # Using 'Z' prefix to avoid names starting with decimals
    sigma_label = f"SIGMA_Z_{str(s_z).replace('.', 'p')}"
    current_sigma_rows = []
    
    print(f"Starting analysis for Redshift Proximity (sigma_z): {s_z}")

    for mass in halos_mocks_mass:
        # 1. Data Prep
        halos_mock_cutted = halos_mock[halos_mock['Mass'] >= mass]
        current_halo_size = len(halos_mock_cutted['Mass'])
        
        # 2. SkyMatch Initialization
        halos = sky_match.SkyMatch(
                    query_data=halos_mock_cutted, 
                    query_coordinates=halo_coordinates,
                    match_data=clusters_mock,
                    match_coordinates=detections_coordinates
                )
        detections = halos.invert_query_match()
        
        # 3. Matching Phase
        halos_matched = halos.match_2d(cosmo, 50, distance_method=sky_match.DistanceMethod.QUERY_RADIUS)
        detections_matched = detections.match_2d(cosmo, 50, distance_method=sky_match.DistanceMethod.MATCH_RADIUS)
        
        # 4. Filtering & Best Match Selection (NOW USING s_z variable)
        # Using a fixed distance mask of 2 Mpc as per your previous snippet
        mask_halos = (halos_matched.filter_mask_by_redshift_proximity(sigma_z=s_z) & 
                      halos_matched.filter_mask_by_distance(2))
        
        unique_halos = halos_matched.select_best(mask=mask_halos, 
                                               selection_criteria=sky_match.SelectionCriteria.DISTANCES, 
                                               more_massive_column='Mass')
        
        mask_detections = (detections_matched.filter_mask_by_redshift_proximity(sigma_z=s_z) & 
                           detections_matched.filter_mask_by_distance(2))
        
        unique_detections = detections_matched.select_best(mask=mask_detections, 
                                                     selection_criteria=sky_match.SelectionCriteria.DISTANCES, 
                                                     more_massive_column='Mass')
        
        # 5. Table Generation
        multiple_halos_table = halos_matched.to_table_complete(mask=mask_halos, query_properties=halo_properties, match_properties=detections_properties)
        multiple_detections_table = detections_matched.to_table_complete(mask=mask_detections, query_properties=detections_properties, match_properties=halo_properties)
        
        unique_halos_table = halos_matched.to_table_best(best=unique_halos, query_properties=halo_properties, match_properties=detections_properties)
        unique_detections_table = detections_matched.to_table_best(best=unique_detections, query_properties=detections_properties, match_properties=halo_properties)
        
        # 6. Optimized Cross-Match phase
        cross_indices = unique_halos.get_cross_match_indices(unique_detections)
        cross = unique_halos_table[np.isin(unique_halos_table['Index'], cross_indices)]
        
        # 7. Metrics Calculation
        m_halo_r = get_ratios(calculate_split_metrics(multiple_halos_table, 'halo'))
        m_clus_r = get_ratios(calculate_split_metrics(multiple_detections_table, 'cluster'))
        u_halo_r = get_ratios(calculate_catalog_metrics(unique_halos_table, halos_mock, 'halo'))
        u_clus_r = get_ratios(calculate_catalog_metrics(unique_detections_table, clusters_mock, 'cluster'))
        c_halo_r = get_ratios(calculate_catalog_metrics(cross, halos_mock, 'halo'))
        c_clus_r = get_ratios(calculate_catalog_metrics(cross, clusters_mock, 'cluster'))

        # 8. Row Construction
        row = {
            'mass_limit': np.exp(mass),
            'halo_size': current_halo_size,
            'sigma_z': s_z,
            'm_halo_TP': m_halo_r['TP'], 'm_halo_FP': m_halo_r['FP'], 'm_halo_TN': m_halo_r['TN'], 'm_halo_FN': m_halo_r['FN'],
            'm_clus_TP': m_clus_r['TP'], 'm_clus_FP': m_clus_r['FP'], 'm_clus_TN': m_clus_r['TN'], 'm_clus_FN': m_clus_r['FN'],
            'u_halo_TP': u_halo_r['TP'], 'u_halo_FP': u_halo_r['FP'], 'u_halo_TN': u_halo_r['TN'], 'u_halo_FN': u_halo_r['FN'],
            'u_clus_TP': u_clus_r['TP'], 'u_clus_FP': u_clus_r['FP'], 'u_clus_TN': u_clus_r['TN'], 'u_clus_FN': u_clus_r['FN'],
            'c_halo_TP': c_halo_r['TP'], 'c_halo_FP': c_halo_r['FP'], 'c_halo_TN': c_halo_r['TN'], 'c_halo_FN': c_halo_r['FN'],
            'c_clus_TP': c_clus_r['TP'], 'c_clus_FP': c_clus_r['FP'], 'c_clus_TN': c_clus_r['TN'], 'c_clus_FN': c_clus_r['FN']
        }
        
        current_sigma_rows.append(row)
        print(f"  > Finished M = {np.exp(mass):.2e}")

    # Store in dictionary
    dfs_by_sigma[sigma_label] = pd.DataFrame(current_sigma_rows)

# --- Save to Multi-HDU FITS ---
hdul_sigma = fits.HDUList([fits.PrimaryHDU()]) 

for label, df in dfs_by_sigma.items():
    hdu = fits.BinTableHDU(Table.from_pandas(df))
    hdu.name = label
    hdul_sigma.append(hdu)

hdul_sigma.writeto('sigma_z_results.fits', overwrite=True)
print("\nAll sigma_z values complete. Results saved to 'sigma_z_results.fits'")